# Importaciones de librerias

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import re
import unicodedata
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# CONFIGURACIÓN GLOBAL

In [3]:

USAR_TFIDF = True                    # True: TF-IDF, False: Count Vectorizer
RANGO_NGRAMAS = (1, 2)              # Rango de n-gramas a considerar
MAXIMO_CARACTERISTICAS = 40000      # Máximo número de características para el vectorizador
TAMANO_LOTE = 64                    # Tamaño del batch para entrenamiento
EPOCAS = 100                         # Número máximo de épocas de entrenamiento
TAZA_APRENDIZAJE = 0.001            # Tasa de aprendizaje para el optimizador
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuración de preprocesamiento
MODO_PREPROCESAMIENTO = "normalizacion" 

# opciones:
# MODO_PREPROCESAMIENTO = "normalizacion_stopwords" 
# MODO_PREPROCESAMIENTO = "normalizacion_stopwords_stem"

# PREPROCESAMIENTO DE TEXTO

In [4]:
def normalizar_texto(texto_entrada, puntuacion=False, acentos=False, 
                    numeros=False, max_duplicados=2):
    """
    Normaliza el texto aplicando diferentes transformaciones
    
    Args:
        texto_entrada: Texto a normalizar
        puntuacion: Si False elimina puntuación
        acentos: Si False elimina acentos
        numeros: Si False elimina números
        max_duplicados: Máximo de caracteres duplicados consecutivos
    """
    # Configuración de caracteres a eliminar
    PUNTUACION = ";:,.\\\\-\\\"'/"
    SIMBOLOS = "()[]¿?¡!{}~<>|"
    NUMEROS = "0123456789"
    SIMBOLOS_SALTAR = set(PUNTUACION + SIMBOLOS)
    
    # Normalización Unicode
    texto_normalizado_unicode = unicodedata.normalize('NFKD', texto_entrada)
    texto_normalizado = []
    caracter_anterior = ''
    contador_duplicados = 0
    
    for caracter in texto_normalizado_unicode:
        # Eliminar números si está configurado
        if not numeros and caracter in NUMEROS:
            continue
            
        # Eliminar puntuación si está configurado
        if not puntuacion and caracter in SIMBOLOS_SALTAR:
            continue
            
        # Eliminar acentos si está configurado
        if not acentos and unicodedata.combining(caracter):
            continue
            
        # Controlar caracteres duplicados
        if caracter_anterior == caracter:
            contador_duplicados += 1
            if contador_duplicados >= max_duplicados:
                continue
        else:
            contador_duplicados = 0
            
        texto_normalizado.append(caracter)
        caracter_anterior = caracter
    
    # Reconstruir texto y normalizar espacios
    texto_final = unicodedata.normalize('NFKD', "".join(texto_normalizado))
    texto_final = re.sub(r'(\\s)+', r' ', texto_final.strip(), flags=re.IGNORECASE)
    
    return texto_final

def fabrica_preprocesamiento(modo):
    """
    Factory que devuelve funciones de preprocesamiento según el modo especificado
    
    Args:
        modo: Tipo de preprocesamiento ('normalizacion', 'normalizacion_stopwords', 
              'normalizacion_stopwords_stem')
    
    Returns:
        Tupla (función_preprocesamiento, función_tokenizador)
    """
    # Configuración de stopwords y stemmer
    STOPWORDS_INGLES = stopwords.words("english")
    stemmer = SnowballStemmer("english")
    
    def preprocesar_texto(texto):
        """Aplica limpieza básica al texto"""
        texto = texto.lower()
        # texto = eliminar_emojis(texto)  # Descomentar si se necesitan eliminar emojis
        texto = re.sub(r"http\\S+|www\\S+|https\\S+", "", texto)  # Eliminar URLs
        texto = re.sub(r"@\\w+", "", texto)  # Eliminar menciones
        texto = normalizar_texto(texto, puntuacion=True)  # Mantener puntuación
        return texto
    
    def tokenizar_texto(texto):
        """Tokeniza el texto según el modo especificado"""
        tokens = word_tokenize(texto)
        
        # Eliminar stopwords para modos avanzados
        if modo in ['normalizacion_stopwords', 'normalizacion_stopwords_stem']:
            tokens = [token for token in tokens if token not in STOPWORDS_INGLES and len(token) > 2]
        
        # Aplicar stemming si está especificado
        if modo == 'normalizacion_stopwords_stem':
            tokens = [stemmer.stem(token) for token in tokens]
            
        return tokens
    
    return preprocesar_texto, tokenizar_texto

# VECTORIZACIÓN DE TEXTO

In [5]:

class VectorizadorTFIDF:
    """Vectorizador usando TF-IDF"""
    
    def __init__(self, modo_preprocesamiento='normalizacion'):
        self.vectorizador_tfidf = None
        self.modo_preprocesamiento = modo_preprocesamiento
    
    def crear_matriz_tfidf(self, datos_entrenamiento, configuracion_ngramas, configuracion_max_features):
        """Crea la matriz TF-IDF a partir de los datos de entrenamiento"""
        preprocesar, tokenizar = fabrica_preprocesamiento(self.modo_preprocesamiento)
        
        self.vectorizador_tfidf = TfidfVectorizer(
            analyzer="word",
            preprocessor=preprocesar,
            tokenizer=tokenizar,
            ngram_range=configuracion_ngramas,
            max_features=configuracion_max_features
        )
        
        matriz_tfidf = self.vectorizador_tfidf.fit_transform(datos_entrenamiento)
        return matriz_tfidf.toarray()
    
    def transformar_matriz_tfidf(self, datos_prueba):
        """Transforma nuevos datos usando el vectorizador ya entrenado"""
        matriz_tfidf = self.vectorizador_tfidf.transform(datos_prueba)
        return matriz_tfidf.toarray()

class VectorizadorConteo:
    """Vectorizador usando Count Vectorizer"""
    
    def __init__(self, modo_preprocesamiento='normalizacion'):
        self.vectorizador_conteo = None
        self.modo_preprocesamiento = modo_preprocesamiento
    
    def crear_matriz_conteo(self, datos_entrenamiento, configuracion_ngramas, configuracion_max_features):
        """Crea la matriz de conteo a partir de los datos de entrenamiento"""
        preprocesar, tokenizar = fabrica_preprocesamiento(self.modo_preprocesamiento)
        
        self.vectorizador_conteo = CountVectorizer(
            analyzer="word",
            preprocessor=preprocesar,
            tokenizer=tokenizar,
            ngram_range=configuracion_ngramas,
            max_features=configuracion_max_features
        )
        
        matriz_conteo = self.vectorizador_conteo.fit_transform(datos_entrenamiento)
        return matriz_conteo.toarray().astype(np.float32)
    
    def transformar_matriz_conteo(self, datos_prueba):
        """Transforma nuevos datos usando el vectorizador ya entrenado"""
        matriz_conteo = self.vectorizador_conteo.transform(datos_prueba)
        return matriz_conteo.toarray().astype(np.float32)

# MODELO DE RED NEURONAL

In [6]:
class RedNeuronalHumor(nn.Module):
    """Red Neuronal Multicapa para clasificacion de humor"""
    
    def __init__(self, tamano_entrada, tamano_salida):
        super().__init__()
        
        # Configuración de capas ocultas
        neuronas_capa1 = 512
        neuronas_capa2 = 128
        neuronas_capa3 = 32
        
        # Capa 1: Entrada -> 512 neuronas
        self.capa_lineal1 = nn.Linear(tamano_entrada, neuronas_capa1)
        self.normalizacion1 = nn.BatchNorm1d(neuronas_capa1)
        self.activacion1 = nn.LeakyReLU()
        self.dropout1 = nn.Dropout(p=0.5)
        
        # Capa 2: 512 -> 128 neuronas
        self.capa_lineal2 = nn.Linear(neuronas_capa1, neuronas_capa2)
        self.normalizacion2 = nn.BatchNorm1d(neuronas_capa2)
        self.activacion2 = nn.LeakyReLU()
        self.dropout2 = nn.Dropout(p=0.4)
        
        # Capa 3: 128 -> 32 neuronas
        self.capa_lineal3 = nn.Linear(neuronas_capa2, neuronas_capa3)
        self.normalizacion3 = nn.BatchNorm1d(neuronas_capa3)
        self.activacion3 = nn.LeakyReLU()
        self.dropout3 = nn.Dropout(p=0.2)
        
        # Capa de salida: 32 -> 2 clases
        self.capa_salida = nn.Linear(neuronas_capa3, tamano_salida)
        
        # Inicialización de pesos
        self._inicializar_pesos()
    
    def _inicializar_pesos(self):
        """Inicializa los pesos de la red usando He initialization"""
        for modulo in self.modules():
            if isinstance(modulo, nn.Linear):
                nn.init.kaiming_normal_(modulo.weight, mode='fan_out', nonlinearity='relu')
                if modulo.bias is not None:
                    nn.init.constant_(modulo.bias, 0)
    
    def forward(self, x):
        """Propagación hacia adelante"""
        # Capa 1
        x = self.capa_lineal1(x)
        x = self.normalizacion1(x)
        x = self.activacion1(x)
        x = self.dropout1(x)
        
        # Capa 2
        x = self.capa_lineal2(x)
        x = self.normalizacion2(x)
        x = self.activacion2(x)
        x = self.dropout2(x)
        
        # Capa 3
        x = self.capa_lineal3(x)
        x = self.normalizacion3(x)
        x = self.activacion3(x)
        x = self.dropout3(x)
        
        # Capa de salida (sin activacion)
        x = self.capa_salida(x)
        
        return x

# FUNCIONES DE "UTILIDAD"

In [7]:
def crear_minilotes(datos_entrada, etiquetas, tamano_lote):
    """
    Crea minilotes para el entrenamiento de la red neuronal
    
    Args:
        datos_entrada: Tensor con datos de entrada
        etiquetas: Tensor con etiquetas
        tamano_lote: Tamaño de cada minilote
    
    Returns:
        DataLoader para iterar sobre los minilotes
    """
    conjunto_datos = TensorDataset(datos_entrada, etiquetas)
    cargador_datos = DataLoader(conjunto_datos, batch_size=tamano_lote, shuffle=True)
    return cargador_datos

In [8]:
def guardar_resultados(predicciones, nombre_archivo):
    """
    Guarda las predicciones en un archivo CSV
    Args:
        predicciones: Array con las predicciones (0 o 1)
        nombre_archivo: Nombre del archivo de salida
    """
    dataframe = pd.DataFrame(predicciones, columns=['klass'])
    dataframe['id'] = dataframe.index + 1
    dataframe = dataframe[['id', 'klass']]
    dataframe.to_csv(nombre_archivo, index=False)
    print(f"Resultados guardados en {nombre_archivo}")

# FUNCIÓN PRINCIPAL DE ENTRENAMIENTO

In [9]:


def entrenar_modelo():
    """Función principal para entrenar el modelo de detección de humor"""
    
    print("Cargar datos de entrenamiento...")
    
    # Cargar datos de entrenamiento
    datos_entrenamiento = pd.read_json("dataset/dataset_humor_train.json", lines=True)
    print("Distribución de clases en entrenamiento:")
    print(datos_entrenamiento.klass.value_counts())
    
    textos_entrenamiento = datos_entrenamiento['text'].to_numpy()
    etiquetas_entrenamiento = datos_entrenamiento['klass'].to_numpy()
    
    # Cargar datos de prueba
    print("Cargar datos de prueba...")
    datos_prueba = pd.read_json("dataset/dataset_humor_test.json", lines=True)
    print("Distribución de clases en prueba:")
    print(datos_prueba.klass.value_counts())
    
    textos_prueba = datos_prueba['text'].to_numpy()
    etiquetas_prueba = datos_prueba['klass'].to_numpy()
    
    # Dividir datos en entrenamiento y validación
    textos_entrenamiento, textos_validacion, etiquetas_entrenamiento, etiquetas_validacion = train_test_split(
        textos_entrenamiento, etiquetas_entrenamiento, 
        test_size=0.1, stratify=etiquetas_entrenamiento, random_state=42
    )
    
    print(f"Vectorizacion con {USAR_TFIDF and 'TF-IDF' or 'COUNT VECTORIZER'}")
    
    # Vectorizar textos
    if USAR_TFIDF:
        vectorizador = VectorizadorTFIDF(modo_preprocesamiento=MODO_PREPROCESAMIENTO)
        matriz_entrenamiento = vectorizador.crear_matriz_tfidf(
            textos_entrenamiento, RANGO_NGRAMAS, MAXIMO_CARACTERISTICAS
        )
        matriz_validacion = vectorizador.transformar_matriz_tfidf(textos_validacion)
        matriz_prueba = vectorizador.transformar_matriz_tfidf(textos_prueba)
    else:
        vectorizador = VectorizadorConteo(modo_preprocesamiento=MODO_PREPROCESAMIENTO)
        matriz_entrenamiento = vectorizador.crear_matriz_conteo(
            textos_entrenamiento, RANGO_NGRAMAS, MAXIMO_CARACTERISTICAS
        )
        matriz_validacion = vectorizador.transformar_matriz_conteo(textos_validacion)
        matriz_prueba = vectorizador.transformar_matriz_conteo(textos_prueba)
    
    print(f"Tamaño matriz entrenamiento: {matriz_entrenamiento.shape}")
    
    # Convertir a tensores de PyTorch
    tensor_entrenamiento_x = torch.from_numpy(matriz_entrenamiento).to(torch.float32)
    tensor_entrenamiento_y = torch.from_numpy(etiquetas_entrenamiento).long()
    
    tensor_validacion_x = torch.from_numpy(matriz_validacion).to(torch.float32)
    
    print(f" Modelo: {DISPOSITIVO}, Entrenando")
    
    # Crear modelo
    tamano_entrada = matriz_entrenamiento.shape[1]
    tamano_salida = 2  # 2 clases: humor vs no humor
    
    modelo = RedNeuronalHumor(tamano_entrada, tamano_salida)
    modelo.to(DISPOSITIVO)
    
    # Calcular pesos para clases desbalanceadas
    clases_unicas = np.unique(etiquetas_entrenamiento)
    pesos_clases = compute_class_weight('balanced', classes=clases_unicas, y=etiquetas_entrenamiento)
    pesos_clases[1] = pesos_clases[1] * 1.3  # Dar más peso a la clase minoritaria (humor)
    pesos_tensor = torch.tensor(pesos_clases).float().to(DISPOSITIVO)
    
    # Configurar función de pérdida y optimizador
    criterio_perdida = nn.CrossEntropyLoss(weight=pesos_tensor)
    optimizador = optim.Adam(modelo.parameters(), lr=TAZA_APRENDIZAJE, weight_decay=1e-4)
    programador_tasa_aprendizaje = optim.lr_scheduler.ReduceLROnPlateau(
        optimizador, mode='min', factor=0.5, patience=3
    )
    
    # Entrenamiento
    for epoca in range(EPOCAS):
        modelo.train()
        perdida_total = 0
        
        print("_" *20)
        cargador_datos = crear_minilotes(tensor_entrenamiento_x, tensor_entrenamiento_y, TAMANO_LOTE)
        
        for datos_lote, etiquetas_lote in cargador_datos:
            # Mover datos al dispositivo (GPU/CPU)
            datos_lote = datos_lote.to(DISPOSITIVO)
            etiquetas_lote = etiquetas_lote.to(DISPOSITIVO)
            
            optimizador.zero_grad()
            
            # Propagación hacia adelante
            predicciones_lote = modelo(datos_lote)
            perdida = criterio_perdida(predicciones_lote, etiquetas_lote)
            perdida_total += perdida.item()
            
            # Retropropagación
            perdida.backward()
            optimizador.step()
            
            # Logging ocasional
            if np.random.random() < 0.01:

                print(f"Lote con perdida: {perdida.item():.4f}")
        
        print(f"Época {epoca+1}/{EPOCAS}, promedio de perdida: {perdida_total/len(cargador_datos):.4f}")
        
        
        # Validación
        modelo.eval()
        with torch.no_grad():
            tensor_validacion_gpu = tensor_validacion_x.to(DISPOSITIVO)
            tensor_etiquetas_validacion = torch.from_numpy(etiquetas_validacion).long().to(DISPOSITIVO)
            
            logits_validacion = modelo(tensor_validacion_gpu)
            perdida_validacion = criterio_perdida(logits_validacion, tensor_etiquetas_validacion)
            
            # Convertir logits a clases
            probabilidades_validacion = torch.softmax(logits_validacion, dim=1)
            clases_predichas_validacion = torch.argmax(probabilidades_validacion, dim=1)
            
            # Mover a CPU para métricas
            clases_predichas_cpu = clases_predichas_validacion.cpu().numpy()
            
            # Calcular métricas
            precision = precision_score(etiquetas_validacion, clases_predichas_cpu, average='macro')
            recall = recall_score(etiquetas_validacion, clases_predichas_cpu, average='macro')
            f1 = f1_score(etiquetas_validacion, clases_predichas_cpu, average='macro')
            exactitud = accuracy_score(etiquetas_validacion, clases_predichas_cpu)
            
            print(f"Época {epoca+1} | Precisión: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | Exactitud: {exactitud:.4f}")
            print("_" * 20)
        programador_tasa_aprendizaje.step(perdida_validacion)
    
    print("Evaluacion finalizada")
    
    # Evaluar en conjunto de validación
    modelo.eval()
    with torch.no_grad():
        tensor_validacion_gpu = tensor_validacion_x.to(DISPOSITIVO)
        logits_finales = modelo(tensor_validacion_gpu)
        clases_finales = torch.argmax(torch.softmax(logits_finales, dim=1), dim=1)
        predicciones_finales = clases_finales.cpu().numpy()
    
    # Matriz de confusión y reporte
    print("Matriz de confusion")
    matriz_confusion = confusion_matrix(etiquetas_validacion, predicciones_finales)
    print(matriz_confusion)
    
    print("\n-Reporte de Clasificación")
    print(classification_report(etiquetas_validacion, predicciones_finales, digits=4, target_names=['No Humor', 'Humor']))
    
    # Generar predicciones para el conjunto de prueba
    print("Predicciones del conjunto de prueba")
    tensor_prueba = torch.from_numpy(matriz_prueba).to(torch.float32)
    tensor_prueba = tensor_prueba.to(DISPOSITIVO)
    
    modelo.eval()
    with torch.no_grad():
        logits_prueba = modelo(tensor_prueba)
        clases_prueba = torch.argmax(logits_prueba, dim=1)
        predicciones_prueba = clases_prueba.cpu().numpy()
    
    print("Predicciones generadas para datos de prueba:")
    print(predicciones_prueba)
    
    # Guardar resultados
    nombre_archivo = f"resultados_humor_{'TFIDF' if USAR_TFIDF else 'Count'}_{MODO_PREPROCESAMIENTO}.csv"
    guardar_resultados(predicciones_prueba, nombre_archivo)
    
    return modelo, vectorizador



## Ejecutamos el modelo de entrenamiento

In [ ]:
# EJECUCIÓN PRINCIPAL
modelo_entrenado, vectorizador_entrenado = entrenar_modelo()